### Setup Environment

In [1]:
!git clone https://github.com/NhuGiap04/Fk-Diffusion-Steering.git

Cloning into 'Fk-Diffusion-Steering'...
remote: Enumerating objects: 338, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 338 (delta 82), reused 72 (delta 48), pack-reused 206 (from 3)
Receiving objects: 100% (338/338), 210.99 MiB | 15.68 MiB/s, done.
Resolving deltas: 100% (168/168), done.


In [2]:
%cd Fk-Diffusion-Steering
!git pull

/content/Fk-Diffusion-Steering
Already up to date.


In [3]:
%pip install -r requirements.txt -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 29.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.1/165.1 kB 16.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 6.4 MB/s eta 0:00:00
     ━━━━

### Evolve Steering

In [ ]:
%cd text_to_image

import os
# os.environ["CUDA_VISIBLE_DEVICES"] = ''
# os.environ['HF_HOME'] = ''

import json
import argparse
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from PIL import Image

import torch
from diffusers import DDIMScheduler

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from diffusers import DDIMScheduler
from evolve_diffusers import BaseSDXL
from evolve_diffusers.steer_pipeline import steer_sample


def visualize_steer_sample_output(result, latent_trajectory, max_images=4):
    """Visualize final steered images and latent trajectory statistics."""
    if hasattr(result, "images") and len(result.images) > 0:
        n = min(max_images, len(result.images))
        fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
        if n == 1:
            axes = [axes]

        for i in range(n):
            axes[i].imshow(result.images[i])
            axes[i].set_title(f"Sample {i + 1}")
            axes[i].axis("off")

        fig.suptitle("Final steered samples", fontsize=14)
        fig.tight_layout()
        plt.show()

    if latent_trajectory is None or len(latent_trajectory) == 0:
        print("No latent trajectory found to visualize.")
        return

    latent_rms = []
    latent_std = []
    for lat in latent_trajectory:
        x = lat.float()
        latent_rms.append(torch.sqrt(torch.mean(x * x)).item())
        latent_std.append(torch.std(x).item())

    denoise_step = np.arange(len(latent_trajectory) - 1, -1, -1)

    fig, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.plot(denoise_step, latent_rms, label="latent RMS")
    ax.plot(denoise_step, latent_std, label="latent std")
    ax.set_xlabel("Denoising step index (high to low)")
    ax.set_ylabel("Value")
    ax.set_title("Steering trajectory diagnostics")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()


device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = BaseSDXL.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)

# accepted clean latent particles from a previous loop: [M, 4, H/8, W/8]
accepted_x0 = torch.randn(4, 4, 128, 128, device=device)

result, latent_trajectory = steer_sample(
    model=pipe,
    prompt=["a photo of a blue clock and a white cup"] * 4,
    accepted_x0=accepted_x0,
    steer_start_timestep=160,
    steer_end_timestep=20,
    stein_step_size=0.04,
    stein_bandwidth=None,
    # guidance_scale defaults to 5.0, matching BaseSDXL default CFG.
    num_inference_steps=50,
    output_type="pil",
)

# result.images contains final generated images.
# latent_trajectory stores one latent tensor per denoising step on CPU.
print(len(result.images), len(latent_trajectory), latent_trajectory[-1].shape)

visualize_steer_sample_output(result, latent_trajectory, max_images=4)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from evolve_diffusers.steer_pipeline import iterative_sample_with_stein


def iterative_stein_loop(
    model,
    prompt,
    *,
    num_loops=4,
    num_particles=4,
    steer_start_timestep=160,
    steer_end_timestep=20,
    base_threshold=0.0,
    stein_step_size=0.04,
    stein_bandwidth=None,
    num_inference_steps=50,
    guidance_scale=5.0,
    guidance_reward_fn="ImageReward",
    max_images_to_show=4,
):
    """Run iterative Stein steering and display loop-level diagnostics + final images."""
    out = iterative_sample_with_stein(
        model=model,
        prompt=prompt,
        num_loops=num_loops,
        num_particles=num_particles,
        steer_start_timestep=steer_start_timestep,
        steer_end_timestep=steer_end_timestep,
        base_threshold=base_threshold,
        stein_step_size=stein_step_size,
        stein_bandwidth=stein_bandwidth,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        guidance_reward_fn=guidance_reward_fn,
    )

    mean_rewards = [float(r.mean().item()) for r in out["rewards"]]
    accepted_counts = [int(a.shape[0]) for a in out["accepted"]]
    thresholds = out["thresholds"]
    loop_idx = np.arange(1, len(mean_rewards) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(loop_idx, mean_rewards, marker="o", label="mean reward")
    axes[0].plot(loop_idx, thresholds, marker="s", label="threshold")
    axes[0].set_xlabel("Loop")
    axes[0].set_ylabel("Score")
    axes[0].set_title("Reward and threshold per loop")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].bar(loop_idx, accepted_counts)
    axes[1].set_xlabel("Loop")
    axes[1].set_ylabel("# accepted particles")
    axes[1].set_title("Accepted particles per loop")
    axes[1].grid(axis="y", alpha=0.3)
    fig.tight_layout()
    plt.show()

    final_images = out["results"][-1].images
    n = min(max_images_to_show, len(final_images))
    fig, ax = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        ax = [ax]

    for i in range(n):
        ax[i].imshow(final_images[i])
        ax[i].axis("off")
        ax[i].set_title(f"Final loop sample {i + 1}")

    fig.suptitle("Iterative Stein loop: final generated samples", fontsize=13)
    fig.tight_layout()
    plt.show()

    return out


if "pipe" not in globals():
    raise RuntimeError("Run the steer_sample setup cell first so `pipe` is defined.")

loop_out = iterative_stein_loop(
    model=pipe,
    prompt="a photo of a blue clock and a white cup",
    num_loops=4,
    num_particles=4,
    steer_start_timestep=160,
    steer_end_timestep=20,
    stein_step_size=0.04,
    guidance_scale=5.0,
)

print("Best mean reward:", loop_out["best_mean_reward"])
print("Num loops returned:", len(loop_out["results"]))